#  Notebook 2: Weather Rain Prediction (Classification)
**Dataset:** Weather Forecast Data (2500 samples, 6 features)

**Goal:** Predict whether it will rain using ML classification models

**Target Accuracy:** 85-95%

**Pipeline:**
1. Load & explore the weather dataset
2. Feature engineering & preprocessing
3. Train multiple models: Random Forest, XGBoost, SVM, KNN
4. Hyperparameter tuning with GridSearchCV
5. Ensemble (Voting Classifier) for best results
6. Evaluate with accuracy, precision, recall, F1, AUC-ROC

## Step 1: Install Dependencies & Upload Dataset

In [ ]:
# Install required libraries
!pip install scikit-learn xgboost pandas numpy matplotlib seaborn -q

# === FOR GOOGLE COLAB: Upload your weather_forecast_data.csv ===
# Uncomment the next 3 lines if running in Google Colab
# from google.colab import files
# print("Upload your weather_forecast_data.csv file")
# uploaded = files.upload()

print(" Dependencies ready!")

## Step 2: Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler, LabelEncoder, PolynomialFeatures
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                              confusion_matrix, classification_report, roc_auc_score, roc_curve)
from sklearn.ensemble import (RandomForestClassifier, GradientBoostingClassifier,
                               VotingClassifier, BaggingClassifier, AdaBoostClassifier,
                               ExtraTreesClassifier)
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier

try:
    from xgboost import XGBClassifier
    HAS_XGB = True
except ImportError:
    HAS_XGB = False
    print(" XGBoost not installed, will skip XGBoost model")

import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
print(" Libraries imported!")

## Step 3: Load & Explore Data

In [ ]:
# Load the dataset
import os

possible_paths = ['weather_forecast_data.csv', '/content/weather_forecast_data.csv']
df = None
for p in possible_paths:
    if os.path.exists(p):
        df = pd.read_csv(p)
        print(f" Loaded from: {p}")
        break

if df is None:
    raise FileNotFoundError("weather_forecast_data.csv not found! Please upload it.")

print(f"\nShape: {df.shape}")
print(f"Columns: {df.columns.tolist()}")
print(f"\nData types:\n{df.dtypes}")
df.head(10)

In [ ]:
# Basic statistics
print("Missing values:")
print(df.isnull().sum())
print(f"\nTarget distribution:")
print(df['Rain'].value_counts())
print(f"\nClass balance: {df['Rain'].value_counts(normalize=True).to_dict()}")
print(f"\nStatistics:")
df.describe()

In [ ]:
# Rich EDA Visualization
fig, axes = plt.subplots(2, 3, figsize=(18, 12))
fig.suptitle('Weather Data -- Exploratory Data Analysis', fontsize=16, fontweight='bold')

numeric_cols = ['Temperature', 'Humidity', 'Wind_Speed', 'Cloud_Cover', 'Pressure']
colors_rain = {'rain': '#FF5722', 'no rain': '#2196F3'}

# 1. Distribution of each feature by Rain class
for i, col in enumerate(numeric_cols):
    ax = axes[i // 3, i % 3]
    for label, color in colors_rain.items():
        subset = df[df['Rain'] == label]
        ax.hist(subset[col], bins=30, alpha=0.6, color=color, label=label, edgecolor='white')
    ax.set_title(f'{col} Distribution by Rain', fontsize=11)
    ax.set_xlabel(col)
    ax.set_ylabel('Count')
    ax.legend()
    ax.grid(alpha=0.3)

# 6. Target class distribution
ax = axes[1, 2]
rain_counts = df['Rain'].value_counts()
bars = ax.bar(rain_counts.index, rain_counts.values, color=['#2196F3', '#FF5722'],
              edgecolor='white', alpha=0.85)
ax.set_title('Target Class Distribution', fontsize=11)
ax.set_ylabel('Count')
for bar, val in zip(bars, rain_counts.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 20,
            f'{val} ({val/len(df)*100:.1f}%)', ha='center', fontweight='bold')
ax.grid(alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('weather_eda.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Correlation heatmap
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Encode Rain for correlation
df_corr = df.copy()
df_corr['Rain_encoded'] = (df_corr['Rain'] == 'rain').astype(int)

corr = df_corr[numeric_cols + ['Rain_encoded']].corr()
sns.heatmap(corr, annot=True, cmap='coolwarm', center=0, ax=axes[0], fmt='.3f')
axes[0].set_title('Feature Correlation Matrix', fontsize=12, fontweight='bold')

# Pairplot-like scatter for top 2 features
for label, color in colors_rain.items():
    subset = df[df['Rain'] == label]
    axes[1].scatter(subset['Humidity'], subset['Cloud_Cover'], c=color,
                    alpha=0.5, label=label, s=30, edgecolors='white', linewidth=0.5)
axes[1].set_xlabel('Humidity'); axes[1].set_ylabel('Cloud_Cover')
axes[1].set_title('Humidity vs Cloud Cover by Rain', fontsize=12, fontweight='bold')
axes[1].legend(); axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('weather_correlation.png', dpi=150, bbox_inches='tight')
plt.show()
print(" EDA complete!")

## Step 4: Feature Engineering & Preprocessing

In [ ]:
# Create interaction and polynomial features
df_feat = df.copy()

# Interaction features
df_feat['Humidity_x_CloudCover'] = df_feat['Humidity'] * df_feat['Cloud_Cover']
df_feat['Humidity_x_Pressure'] = df_feat['Humidity'] * df_feat['Pressure']
df_feat['Temp_x_Humidity'] = df_feat['Temperature'] * df_feat['Humidity']
df_feat['Wind_x_Pressure'] = df_feat['Wind_Speed'] * df_feat['Pressure']
df_feat['Cloud_x_Pressure'] = df_feat['Cloud_Cover'] * df_feat['Pressure']

# Ratio features
df_feat['Humidity_ratio'] = df_feat['Humidity'] / (df_feat['Temperature'] + 1)
df_feat['Cloud_Humidity_ratio'] = df_feat['Cloud_Cover'] / (df_feat['Humidity'] + 1)
df_feat['Pressure_deviation'] = np.abs(df_feat['Pressure'] - df_feat['Pressure'].mean())

# Squared features (for non-linear patterns)
df_feat['Humidity_sq'] = df_feat['Humidity'] ** 2
df_feat['Cloud_sq'] = df_feat['Cloud_Cover'] ** 2
df_feat['Temp_sq'] = df_feat['Temperature'] ** 2

# Binned features
df_feat['Humidity_bin'] = pd.qcut(df_feat['Humidity'], q=5, labels=False)
df_feat['Temp_bin'] = pd.qcut(df_feat['Temperature'], q=5, labels=False)

# Encode target
le = LabelEncoder()
df_feat['Rain_label'] = le.fit_transform(df_feat['Rain'])  # 0=no rain, 1=rain
print(f"Label mapping: {dict(zip(le.classes_, le.transform(le.classes_)))}")

# Define feature columns
feature_cols = [c for c in df_feat.columns if c not in ['Rain', 'Rain_label']]
print(f"\nTotal features: {len(feature_cols)}")
print(f"Features: {feature_cols}")

In [ ]:
# Prepare X and y
X = df_feat[feature_cols].values
y = df_feat['Rain_label'].values

# Train/test split (stratified to maintain class balance)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Training set: {X_train_scaled.shape}  |  Test set: {X_test_scaled.shape}")
print(f"Train class balance: {np.bincount(y_train)}")
print(f"Test class balance:  {np.bincount(y_test)}")

## Step 5: Train Multiple Models

In [ ]:
# Define models
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'KNN': KNeighborsClassifier(n_neighbors=7),
    'Decision Tree': DecisionTreeClassifier(max_depth=10, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=200, max_depth=15, random_state=42),
    'Extra Trees': ExtraTreesClassifier(n_estimators=200, max_depth=15, random_state=42),
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=200, max_depth=5,
                                                     learning_rate=0.1, random_state=42),
    'AdaBoost': AdaBoostClassifier(n_estimators=200, learning_rate=0.1, random_state=42),
    'SVM (RBF)': SVC(kernel='rbf', probability=True, random_state=42),
}

if HAS_XGB:
    models['XGBoost'] = XGBClassifier(
        n_estimators=300, max_depth=6, learning_rate=0.1,
        subsample=0.8, colsample_bytree=0.8, random_state=42,
        use_label_encoder=False, eval_metric='logloss'
    )

# Train and evaluate each model
results = {}
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

print(f"{'Model':<25} {'CV Accuracy':<15} {'Test Accuracy':<15} {'F1 Score':<12} {'AUC-ROC':<10}")
print("=" * 77)

for name, model in models.items():
    # Cross-validation scores
    cv_scores = cross_val_score(model, X_train_scaled, y_train, cv=cv, scoring='accuracy')
    
    # Train on full training set
    model.fit(X_train_scaled, y_train)
    
    # Test predictions
    y_pred = model.predict(X_test_scaled)
    y_prob = model.predict_proba(X_test_scaled)[:, 1]
    
    acc = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_prob)
    
    results[name] = {
        'model': model,
        'cv_accuracy': cv_scores.mean(),
        'test_accuracy': acc,
        'f1': f1,
        'auc': auc,
        'y_pred': y_pred,
        'y_prob': y_prob
    }
    
    print(f"{name:<25} {cv_scores.mean():.4f}±{cv_scores.std():.4f}  {acc:.4f}         {f1:.4f}       {auc:.4f}")

print("\n All models trained!")

## Step 6: Hyperparameter Tuning (Best Models)

In [ ]:
# Tune the top performing models
print(" Hyperparameter tuning...\n")

# 1. Random Forest Tuning
rf_params = {
    'n_estimators': [200, 300, 500],
    'max_depth': [10, 15, 20, None],
    'min_samples_split': [2, 5],
    'min_samples_leaf': [1, 2],
    'max_features': ['sqrt', 'log2']
}

rf_grid = GridSearchCV(
    RandomForestClassifier(random_state=42),
    rf_params, cv=5, scoring='accuracy', n_jobs=-1, verbose=0
)
rf_grid.fit(X_train_scaled, y_train)
print(f"Best RF params: {rf_grid.best_params_}")
print(f"Best RF CV accuracy: {rf_grid.best_score_:.4f}")

# 2. Gradient Boosting Tuning
gb_params = {
    'n_estimators': [200, 300, 500],
    'max_depth': [3, 5, 7],
    'learning_rate': [0.05, 0.1, 0.15],
    'subsample': [0.8, 0.9, 1.0],
    'min_samples_split': [2, 5]
}

gb_grid = GridSearchCV(
    GradientBoostingClassifier(random_state=42),
    gb_params, cv=5, scoring='accuracy', n_jobs=-1, verbose=0
)
gb_grid.fit(X_train_scaled, y_train)
print(f"\nBest GB params: {gb_grid.best_params_}")
print(f"Best GB CV accuracy: {gb_grid.best_score_:.4f}")

# 3. XGBoost Tuning (if available)
if HAS_XGB:
    xgb_params = {
        'n_estimators': [200, 300, 500],
        'max_depth': [4, 6, 8],
        'learning_rate': [0.05, 0.1, 0.15],
        'subsample': [0.7, 0.8, 0.9],
        'colsample_bytree': [0.7, 0.8, 0.9],
        'reg_alpha': [0, 0.1],
        'reg_lambda': [1, 1.5]
    }

    xgb_grid = GridSearchCV(
        XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42),
        xgb_params, cv=5, scoring='accuracy', n_jobs=-1, verbose=0
    )
    xgb_grid.fit(X_train_scaled, y_train)
    print(f"\nBest XGB params: {xgb_grid.best_params_}")
    print(f"Best XGB CV accuracy: {xgb_grid.best_score_:.4f}")

print("\n Hyperparameter tuning complete!")

## Step 7: Ensemble Voting Classifier

In [ ]:
# Build an ensemble of the best tuned models
estimators = [
    ('rf', rf_grid.best_estimator_),
    ('gb', gb_grid.best_estimator_),
    ('svm', SVC(kernel='rbf', probability=True, random_state=42)),
    ('et', ExtraTreesClassifier(n_estimators=300, max_depth=15, random_state=42)),
]

if HAS_XGB:
    estimators.append(('xgb', xgb_grid.best_estimator_))

# Soft voting uses predicted probabilities -- generally better
ensemble = VotingClassifier(estimators=estimators, voting='soft')
ensemble.fit(X_train_scaled, y_train)

# Evaluate ensemble
y_pred_ensemble = ensemble.predict(X_test_scaled)
y_prob_ensemble = ensemble.predict_proba(X_test_scaled)[:, 1]

acc_ensemble = accuracy_score(y_test, y_pred_ensemble)
f1_ensemble = f1_score(y_test, y_pred_ensemble)
auc_ensemble = roc_auc_score(y_test, y_prob_ensemble)
prec_ensemble = precision_score(y_test, y_pred_ensemble)
rec_ensemble = recall_score(y_test, y_pred_ensemble)

print("=" * 55)
print("     ENSEMBLE VOTING CLASSIFIER -- EVALUATION")
print("=" * 55)
print(f"  Accuracy   : {acc_ensemble:.4f}  ({acc_ensemble*100:.2f}%)")
print(f"  Precision  : {prec_ensemble:.4f}")
print(f"  Recall     : {rec_ensemble:.4f}")
print(f"  F1 Score   : {f1_ensemble:.4f}")
print(f"  AUC-ROC    : {auc_ensemble:.4f}")
print("=" * 55)

if acc_ensemble >= 0.85:
    print(f"\n  TARGET ACHIEVED! Accuracy = {acc_ensemble*100:.2f}% ≥ 85%")
else:
    print(f"\n Accuracy = {acc_ensemble*100:.2f}% -- below 85% target")

print(f"\nClassification Report:")
print(classification_report(y_test, y_pred_ensemble, target_names=le.classes_))

## Step 8: Visualization -- Results & Analysis

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 14))
fig.suptitle('Weather Prediction -- Model Evaluation', fontsize=16, fontweight='bold')

# 1. Model comparison bar chart
ax = axes[0, 0]
model_names = list(results.keys()) + ['Ensemble']
accuracies = [results[m]['test_accuracy'] for m in results.keys()] + [acc_ensemble]
colors = ['#2196F3'] * len(results) + ['#FF5722']
bars = ax.barh(model_names, accuracies, color=colors, edgecolor='white', alpha=0.85)
ax.set_xlim(0.5, 1.0)
ax.axvline(x=0.85, color='green', linestyle='--', linewidth=2, label='Target (85%)')
for bar, acc in zip(bars, accuracies):
    ax.text(bar.get_width() + 0.005, bar.get_y() + bar.get_height()/2,
            f'{acc:.2%}', va='center', fontweight='bold', fontsize=10)
ax.set_title('Model Accuracy Comparison', fontsize=12, fontweight='bold')
ax.set_xlabel('Accuracy')
ax.legend(fontsize=10)
ax.grid(alpha=0.3, axis='x')

# 2. Confusion Matrix (Ensemble)
ax = axes[0, 1]
cm = confusion_matrix(y_test, y_pred_ensemble)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
            xticklabels=le.classes_, yticklabels=le.classes_)
ax.set_title('Ensemble -- Confusion Matrix', fontsize=12, fontweight='bold')
ax.set_xlabel('Predicted'); ax.set_ylabel('Actual')

# 3. ROC Curves for all models
ax = axes[1, 0]
for name, res in results.items():
    fpr, tpr, _ = roc_curve(y_test, res['y_prob'])
    ax.plot(fpr, tpr, label=f"{name} (AUC={res['auc']:.3f})", alpha=0.7)

# Ensemble ROC
fpr_e, tpr_e, _ = roc_curve(y_test, y_prob_ensemble)
ax.plot(fpr_e, tpr_e, label=f"Ensemble (AUC={auc_ensemble:.3f})",
        color='red', linewidth=3, linestyle='--')
ax.plot([0, 1], [0, 1], 'k--', alpha=0.3)
ax.set_title('ROC Curves', fontsize=12, fontweight='bold')
ax.set_xlabel('False Positive Rate'); ax.set_ylabel('True Positive Rate')
ax.legend(fontsize=8, loc='lower right'); ax.grid(alpha=0.3)

# 4. Feature importance (from best RF)
ax = axes[1, 1]
if hasattr(rf_grid.best_estimator_, 'feature_importances_'):
    importances = rf_grid.best_estimator_.feature_importances_
    feat_imp = pd.Series(importances, index=feature_cols).sort_values(ascending=True)
    feat_imp.tail(15).plot(kind='barh', ax=ax, color='#9C27B0', edgecolor='white', alpha=0.85)
    ax.set_title('Top 15 Feature Importances (RF)', fontsize=12, fontweight='bold')
    ax.set_xlabel('Importance')
    ax.grid(alpha=0.3, axis='x')

plt.tight_layout()
plt.savefig('weather_evaluation.png', dpi=150, bbox_inches='tight')
plt.show()
print(" Evaluation plots saved!")

In [ ]:
# Final Summary Table
print("\n" + "=" * 65)
print("              FINAL MODEL SUMMARY -- WEATHER PREDICTION")
print("=" * 65)
print(f"  Best Single Model    : {max(results, key=lambda k: results[k]['test_accuracy'])}")
print(f"  Best Single Accuracy : {max(r['test_accuracy'] for r in results.values()):.4f}")
print(f"  -----------------------------------------")
print(f"  Ensemble Approach    : Soft Voting ({len(estimators)} models)")
print(f"  Ensemble Accuracy    : {acc_ensemble:.4f} ({acc_ensemble*100:.2f}%)")
print(f"  Ensemble F1 Score    : {f1_ensemble:.4f}")
print(f"  Ensemble AUC-ROC     : {auc_ensemble:.4f}")
print(f"  Ensemble Precision   : {prec_ensemble:.4f}")
print(f"  Ensemble Recall      : {rec_ensemble:.4f}")
print(f"  -----------------------------------------")
print(f"  Dataset              : {len(df)} samples, {len(feature_cols)} features")
print(f"  Train/Test Split     : 80/20 (stratified)")
print(f"  Feature Engineering  : Interactions, ratios, polynomial")
print("=" * 65)

# Save the best model
import pickle
with open('weather_model.pkl', 'wb') as f:
    pickle.dump({'model': ensemble, 'scaler': scaler, 'features': feature_cols}, f)
print("\n Model saved to weather_model.pkl")